# Anomaly Detection

This notebook detects abnormal electricity consumption patterns from the Silver layer.

The anomalies detected are:

- Spike Detection
- Zero Consumption Detection
- Usage Pattern Deviation

The final output is a business-ready anomaly alerts table.

In [0]:
silver_df = spark.table("workspace.default.silver_meter_readings")

In [0]:
display(silver_df)

meter_id,household_id,timestamp,units_consumed,_ingestion_time,is_valid,city,house_type,avg_daily_consumption
M009,H009,2026-04-01T01:00:00.000Z,0.35,2026-07-15T13:23:43.674Z,true,Pune,Independent,6
M010,H010,2026-04-01T03:00:00.000Z,0.1,2026-07-15T13:23:43.674Z,true,Hyderabad,Independent,13
M005,H005,2026-04-01T06:00:00.000Z,4.72,2026-07-15T13:23:43.674Z,true,Kolkata,Apartment,7
M010,H010,2026-04-01T06:00:00.000Z,1.8,2026-07-15T13:23:43.674Z,true,Hyderabad,Independent,13
M001,H001,2026-04-01T09:00:00.000Z,3.87,2026-07-15T13:23:43.674Z,true,Jaipur,Independent,8
M004,H004,2026-04-01T09:00:00.000Z,2.8,2026-07-15T13:23:43.674Z,true,Jaipur,Apartment,15
M002,H002,2026-04-01T10:00:00.000Z,1.06,2026-07-15T13:23:43.674Z,true,Kolkata,Independent,10
M004,H004,2026-04-01T11:00:00.000Z,1.4,2026-07-15T13:23:43.674Z,true,Jaipur,Apartment,15
M002,H002,2026-04-01T13:00:00.000Z,0.68,2026-07-15T13:23:43.674Z,true,Kolkata,Independent,10
M005,H005,2026-04-01T15:00:00.000Z,0.0,2026-07-15T13:23:43.674Z,true,Kolkata,Apartment,7


In [0]:
silver_df.printSchema()

root
 |-- meter_id: string (nullable = true)
 |-- household_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- units_consumed: double (nullable = true)
 |-- _ingestion_time: timestamp (nullable = true)
 |-- is_valid: boolean (nullable = true)
 |-- city: string (nullable = true)
 |-- house_type: string (nullable = true)
 |-- avg_daily_consumption: integer (nullable = true)



In [0]:
silver_df.count()

1400

## Spike Detection

A spike is detected when the electricity consumption exceeds three times the rolling average consumption for the same smart meter.

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import avg, col, when

In [0]:
rolling_window = Window.partitionBy(
    "meter_id"
).orderBy(
    "timestamp"
).rowsBetween(-6, 0)

In [0]:
spike_df = silver_df.withColumn(
    "rolling_avg",
    avg("units_consumed").over(rolling_window)
)

In [0]:
spike_df = spike_df.withColumn(
    "is_spike",
    when(
        col("units_consumed") > 3 * col("rolling_avg"),
        True
    ).otherwise(False)
)

In [0]:
display(
    spike_df.select(
        "meter_id",
        "timestamp",
        "units_consumed",
        "rolling_avg",
        "is_spike"
    )
)

meter_id,timestamp,units_consumed,rolling_avg,is_spike
M001,2026-04-01T00:00:00.000Z,0.41,0.41,false
M001,2026-04-01T01:00:00.000Z,0.8,0.605,false
M001,2026-04-01T02:00:00.000Z,0.41,0.5399999999999999,false
M001,2026-04-01T03:00:00.000Z,0.96,0.645,false
M001,2026-04-01T04:00:00.000Z,0.72,0.6599999999999999,false
M001,2026-04-01T05:00:00.000Z,2.38,0.9466666666666667,false
M001,2026-04-01T06:00:00.000Z,4.82,1.5,true
M001,2026-04-01T07:00:00.000Z,4.16,2.0357142857142856,false
M001,2026-04-01T08:00:00.000Z,4.21,2.5228571428571427,false
M001,2026-04-01T09:00:00.000Z,3.87,3.0171428571428573,false


In [0]:
spike_df.filter(col("is_spike") == True).count()

49

## Zero Consumption Detection

This section detects meters that report zero electricity consumption for three or more consecutive hours.

Such patterns may indicate that the meter is offline or there is a power outage.

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, when, sum

In [0]:
zero_window = Window.partitionBy("meter_id").orderBy("timestamp")

In [0]:
zero_df = spike_df.withColumn(
    "is_zero",
    when(col("units_consumed") == 0, 1).otherwise(0)
)

In [0]:
zero_df = zero_df.withColumn(
    "zero_count",
    sum("is_zero").over(
        zero_window.rowsBetween(-2, 0)
    )
)

In [0]:
zero_df = zero_df.withColumn(
    "is_zero_extended",
    when(col("zero_count") == 3, True).otherwise(False)
)

In [0]:
display(
    zero_df.select(
        "meter_id",
        "timestamp",
        "units_consumed",
        "zero_count",
        "is_zero_extended"
    )
)

meter_id,timestamp,units_consumed,zero_count,is_zero_extended
M001,2026-04-01T00:00:00.000Z,0.41,0,false
M001,2026-04-01T01:00:00.000Z,0.8,0,false
M001,2026-04-01T02:00:00.000Z,0.41,0,false
M001,2026-04-01T03:00:00.000Z,0.96,0,false
M001,2026-04-01T04:00:00.000Z,0.72,0,false
M001,2026-04-01T05:00:00.000Z,2.38,0,false
M001,2026-04-01T06:00:00.000Z,4.82,0,false
M001,2026-04-01T07:00:00.000Z,4.16,0,false
M001,2026-04-01T08:00:00.000Z,4.21,0,false
M001,2026-04-01T09:00:00.000Z,3.87,0,false


In [0]:
zero_df.filter(col("is_zero_extended") == True).count()

0

## Usage Pattern Deviation Detection

This section detects abnormal electricity consumption by comparing each reading with the normal consumption pattern.

A reading is marked as a deviation if it lies outside:

Mean ± 2 × Standard Deviation

In [0]:
from pyspark.sql.functions import avg, stddev

In [0]:
stats_df = zero_df.groupBy("meter_id").agg(
    avg("units_consumed").alias("mean_units"),
    stddev("units_consumed").alias("std_units")
)

In [0]:
deviation_df = zero_df.join(
    stats_df,
    on="meter_id",
    how="left"
)

In [0]:
from pyspark.sql.functions import col, when

deviation_df = deviation_df.withColumn(
    "is_deviation",
    when(
        (col("units_consumed") > col("mean_units") + 2 * col("std_units")) |
        (col("units_consumed") < col("mean_units") - 2 * col("std_units")),
        True
    ).otherwise(False)
)

In [0]:
display(
    deviation_df.select(
        "meter_id",
        "timestamp",
        "units_consumed",
        "mean_units",
        "std_units",
        "is_deviation"
    )
)

meter_id,timestamp,units_consumed,mean_units,std_units,is_deviation
M009,2026-04-01T01:00:00.000Z,0.35,2.2099270072992687,1.720264812185803,false
M010,2026-04-01T03:00:00.000Z,0.1,1.0897080291970807,1.159610064789681,false
M005,2026-04-01T06:00:00.000Z,4.72,3.401582733812951,3.351189980946376,false
M010,2026-04-01T06:00:00.000Z,1.8,1.0897080291970807,1.159610064789681,false
M001,2026-04-01T09:00:00.000Z,3.87,2.8553284671532846,1.9243429173031608,false
M004,2026-04-01T09:00:00.000Z,2.8,1.6267142857142844,1.2906411561851945,false
M002,2026-04-01T10:00:00.000Z,1.06,1.0473188405797103,0.693230604720589,false
M004,2026-04-01T11:00:00.000Z,1.4,1.6267142857142844,1.2906411561851945,false
M002,2026-04-01T13:00:00.000Z,0.68,1.0473188405797103,0.693230604720589,false
M005,2026-04-01T15:00:00.000Z,0.0,3.401582733812951,3.351189980946376,false


In [0]:
deviation_df.filter(col("is_deviation") == True).count()

21

## Anomaly Alerts Table

This section creates the final anomaly alerts table by combining all detected anomaly types.

The output includes:
- Spike Detection
- Zero Consumption Detection
- Usage Pattern Deviation
- Severity

In [0]:
from pyspark.sql.functions import when, lit

alerts_df = deviation_df.withColumn(
    "anomaly_type",
    when(col("is_spike") == True, "SPIKE")
    .when(col("is_zero_extended") == True, "ZERO_EXTENDED")
    .when(col("is_deviation") == True, "DEVIATION")
    .otherwise("NORMAL")
)

In [0]:
alerts_df = alerts_df.withColumn(
    "expected_value",
    col("rolling_avg")
)

In [0]:
from pyspark.sql.functions import abs

alerts_df = alerts_df.withColumn(
    "deviation_pct",
    abs(
        (col("units_consumed") - col("expected_value"))
        / col("expected_value")
    ) * 100
)

In [0]:
alerts_df = alerts_df.withColumn(
    "severity",
    when(col("deviation_pct") >= 100, "HIGH")
    .when(col("deviation_pct") >= 50, "MEDIUM")
    .otherwise("LOW")
)

In [0]:
alerts_df = alerts_df.select(
    "meter_id",
    "household_id",
    "timestamp",
    "units_consumed",
    "expected_value",
    "anomaly_type",
    "deviation_pct",
    "severity"
)

In [0]:
display(alerts_df)

meter_id,household_id,timestamp,units_consumed,expected_value,anomaly_type,deviation_pct,severity
M001,H001,2026-04-01T00:00:00.000Z,0.41,0.41,NORMAL,0.0,LOW
M001,H001,2026-04-01T01:00:00.000Z,0.8,0.605,NORMAL,32.23140495867769,LOW
M001,H001,2026-04-01T02:00:00.000Z,0.41,0.5399999999999999,NORMAL,24.074074074074066,LOW
M001,H001,2026-04-01T03:00:00.000Z,0.96,0.645,NORMAL,48.837209302325576,LOW
M001,H001,2026-04-01T04:00:00.000Z,0.72,0.6599999999999999,NORMAL,9.090909090909099,LOW
M001,H001,2026-04-01T05:00:00.000Z,2.38,0.9466666666666667,NORMAL,151.40845070422532,HIGH
M001,H001,2026-04-01T06:00:00.000Z,4.82,1.5,SPIKE,221.33333333333334,HIGH
M001,H001,2026-04-01T07:00:00.000Z,4.16,2.0357142857142856,NORMAL,104.35087719298248,HIGH
M001,H001,2026-04-01T08:00:00.000Z,4.21,2.5228571428571427,NORMAL,66.87429218573048,MEDIUM
M001,H001,2026-04-01T09:00:00.000Z,3.87,3.0171428571428573,NORMAL,28.267045454545446,LOW


In [0]:
alerts_df.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("workspace.default.gold_anomaly_alerts")

In [0]:
%sql

SELECT * FROM workspace.default.gold_anomaly_alerts;

meter_id,household_id,timestamp,units_consumed,expected_value,anomaly_type,deviation_pct,severity
M001,H001,2026-04-01T00:00:00.000Z,0.41,0.41,NORMAL,0.0,LOW
M001,H001,2026-04-01T01:00:00.000Z,0.8,0.605,NORMAL,32.23140495867769,LOW
M001,H001,2026-04-01T02:00:00.000Z,0.41,0.5399999999999999,NORMAL,24.074074074074066,LOW
M001,H001,2026-04-01T03:00:00.000Z,0.96,0.645,NORMAL,48.837209302325576,LOW
M001,H001,2026-04-01T04:00:00.000Z,0.72,0.6599999999999999,NORMAL,9.090909090909099,LOW
M001,H001,2026-04-01T05:00:00.000Z,2.38,0.9466666666666667,NORMAL,151.40845070422532,HIGH
M001,H001,2026-04-01T06:00:00.000Z,4.82,1.5,SPIKE,221.33333333333334,HIGH
M001,H001,2026-04-01T07:00:00.000Z,4.16,2.0357142857142856,NORMAL,104.35087719298248,HIGH
M001,H001,2026-04-01T08:00:00.000Z,4.21,2.5228571428571427,NORMAL,66.87429218573048,MEDIUM
M001,H001,2026-04-01T09:00:00.000Z,3.87,3.0171428571428573,NORMAL,28.267045454545446,LOW


In [0]:
display(alerts_df)

meter_id,household_id,timestamp,units_consumed,expected_value,anomaly_type,deviation_pct,severity
M001,H001,2026-04-01T00:00:00.000Z,0.41,0.41,NORMAL,0.0,LOW
M001,H001,2026-04-01T01:00:00.000Z,0.8,0.605,NORMAL,32.23140495867769,LOW
M001,H001,2026-04-01T02:00:00.000Z,0.41,0.5399999999999999,NORMAL,24.074074074074066,LOW
M001,H001,2026-04-01T03:00:00.000Z,0.96,0.645,NORMAL,48.837209302325576,LOW
M001,H001,2026-04-01T04:00:00.000Z,0.72,0.6599999999999999,NORMAL,9.090909090909099,LOW
M001,H001,2026-04-01T05:00:00.000Z,2.38,0.9466666666666667,NORMAL,151.40845070422532,HIGH
M001,H001,2026-04-01T06:00:00.000Z,4.82,1.5,SPIKE,221.33333333333334,HIGH
M001,H001,2026-04-01T07:00:00.000Z,4.16,2.0357142857142856,NORMAL,104.35087719298248,HIGH
M001,H001,2026-04-01T08:00:00.000Z,4.21,2.5228571428571427,NORMAL,66.87429218573048,MEDIUM
M001,H001,2026-04-01T09:00:00.000Z,3.87,3.0171428571428573,NORMAL,28.267045454545446,LOW
